## Imports
Libraries for data manipulation, preprocessing, clustering and statistics.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re, unicodedata
import json
import sklearn

from scipy import stats
from scipy.spatial.distance import cdist
from matplotlib.colors import LinearSegmentedColormap
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples
from sklearn.decomposition import PCA

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 800
pd.set_option("display.max_columns", 50)

## Data loading and cleaning
Loads the mobility dataset, harmonizes country names, and normalizes the *Fewer Opportunities* column into Yes/No/Unknown.

In [3]:
mob = pd.read_csv("./Datasets/Processed/df_mobility_hicp.csv")

def clean_fewer_opportunities(v):
    v = str(v).strip().lower()
    return {"yes": "Yes", "1": "Yes", "no": "No", "0": "No"}.get(v, "Unknown")

mob["Fewer_Opportunities_clean"] = mob["Fewer Opportunities"].apply(clean_fewer_opportunities)
mob.drop(columns=["Fewer Opportunities"], inplace=True)
mob.rename(columns={"Fewer_Opportunities_clean": "Fewer Opportunities"}, inplace=True)
mob.head()

C:\Users\rpasq\AppData\Local\Temp\ipykernel_3760\690240716.py:1: DtypeWarning: Columns (0: Fewer Opportunities) have mixed types. Specify dtype option on import or set low_memory=False.
  mob = pd.read_csv("./Datasets/Processed/df_mobility_hicp.csv")


,Academic Year,Mobility Duration,Field of Education,Participant Country,Education Level,Participant Gender,Participant Age,Sending Country,Sending Country HICP,Sending City,Sending Organization,Receiving Country,Receiving Country HICP,Receiving City,Receiving Organization,Fewer Opportunities
0,2023,61,Business and administration,Austria,ISCED-6 - First cycle / Bachelor’s or equivale...,Female,22,Austria,130.4,WIEN,WU,Germany,125.9,Munich,-,No
1,2023,101,Economics,Germany,ISCED-7 - Second cycle / Master’s or equivalen...,Male,27,Austria,130.4,WIEN,WU,France,120.5,Toulouse,Université Toulouse Capitole,No
2,2023,121,Business and administration,Austria,ISCED-6 - First cycle / Bachelor’s or equivale...,Female,22,Austria,130.4,WIEN,WU,France,120.5,Paris,-,No
3,2023,171,Business and administration,Germany,ISCED-6 - First cycle / Bachelor’s or equivale...,Male,22,Austria,130.4,WIEN,WU,France,120.5,Paris,-,No
4,2023,86,Business and administration,Austria,ISCED-6 - First cycle / Bachelor’s or equivale...,Male,23,Austria,130.4,WIEN,WU,Germany,125.9,Hamburg,-,No


## Feature engineering — field of study
Maps raw *Field of Education* values onto the 10 ISCED-F macro-fields, to avoid an overly sparse one-hot encoding.


In [5]:
with open('./mapping_results.json', 'r') as map:
    mapping_data = json.load(map)

In [5]:
mob.shape

(3172958, 16)

## Feature engineering — HICP and country frequency
Winsorizes the HICP index (to limit the effect of Türkiye-related outliers), computes the sending/receiving differential, and the frequency of each country in the sample.

In [6]:
mob2 = mob.copy()
mob2["HICP diff"] = mob2["Receiving Country HICP"] - mob2["Sending Country HICP"]

mob2.head()

,Academic Year,Mobility Duration,Field of Education,Participant Country,Education Level,Participant Gender,Participant Age,Sending Country,Sending Country HICP,Sending City,Sending Organization,Receiving Country,Receiving Country HICP,Receiving City,Receiving Organization,Fewer Opportunities,HICP diff,Sending Country Freq,Receiving Country Freq
0,2023,61,Business and administration,Austria,ISCED-6 - First cycle / Bachelor’s or equivale...,Female,22,Austria,130.4,WIEN,WU,Germany,125.9,Munich,-,No,-4.5,0.022684,0.099079
1,2023,101,Economics,Germany,ISCED-7 - Second cycle / Master’s or equivalen...,Male,27,Austria,130.4,WIEN,WU,France,120.5,Toulouse,Université Toulouse Capitole,No,-9.9,0.022684,0.089758
2,2023,121,Business and administration,Austria,ISCED-6 - First cycle / Bachelor’s or equivale...,Female,22,Austria,130.4,WIEN,WU,France,120.5,Paris,-,No,-9.9,0.022684,0.089758
3,2023,171,Business and administration,Germany,ISCED-6 - First cycle / Bachelor’s or equivale...,Male,22,Austria,130.4,WIEN,WU,France,120.5,Paris,-,No,-9.9,0.022684,0.089758
4,2023,86,Business and administration,Austria,ISCED-6 - First cycle / Bachelor’s or equivale...,Male,23,Austria,130.4,WIEN,WU,Germany,125.9,Hamburg,-,No,-4.5,0.022684,0.099079


## Split into categorical and numerical columns

In [ ]:
student_num_cols = ["Mobility Duration", "Participant Age", "Sending Country HICP", "Receiving Country HICP", "Sending Institution Rank", "Receiving Institution Rank"]
student_cat_cols = ["Participant Gender", "Fewer Opportunities", "Education Level", "Field of Education"]


# features used for external validation >:3
eta_cols = student_num_cols + ["HICP diff"]
mob2[eta_cols] = mob2[eta_cols].fillna(mob2[eta_cols].median())
print(mob2[eta_cols].isna().sum())

Mobility Duration         0
Participant Age           0
Sending Country HICP      0
Receiving Country HICP    0
HICP diff                 0
Sending Country Freq      0
Receiving Country Freq    0
dtype: int64


## Weighted preprocessing pipeline
Defines a function that builds the feature matrix by scaling/one-hot-encoding each block with a configurable weight, and four weight combinations to compare: baseline, geo-economic emphasis, demographic/field emphasis, and an **equity/didactic focus** that removes the geographic blocks entirely (weight 0) to check whether a socio-didactic pattern exists independently of geography.

In [8]:
mob3 = mob2.sample(frac=0.005, random_state=42).reset_index(drop=True)
mob3.shape

(15865, 19)

In [ ]:
from scipy.sparse import issparse

def build_matrix(weights):
    ct = ColumnTransformer([
        ("student_num", StandardScaler(), student_num_cols),
        ("student_cat", OneHotEncoder(handle_unknown="infrequent_if_exist"), student_cat_cols),
    ], transformer_weights=weights)

    Xt = ct.fit_transform(mob3)
    if issparse(Xt):
        Xt = Xt.toarray()

    feature_names = ct.get_feature_names_out()
    return Xt, feature_names

version = {
    "baseline": {"student_num": 1, "student_cat": 1},
}

## Elbow and silhouette methods
Scans k=2, ..., 10 with elbow and silhouette, automatically picks the best k (excluding k=2, which only isolates outliers) and trains the final KMeans.

In [ ]:
name = "baseline"
w = version[name]
Xt, feat_names = build_matrix(w)
inertias, sils = [], []
k_range = range(2, 11)
for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(Xt)
    inertias.append(km.inertia_)
    sils.append(silhouette_score(Xt, labels))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(list(k_range), inertias, marker="o", color="#591427")
axes[0].set_title(f"{name}: elbow")
axes[0].set_xlabel("k"); axes[0].set_ylabel("inertia")

axes[1].plot(list(k_range), sils, marker="o", color="#591427")
axes[1].set_title(f"{name}: silhouette")
axes[1].set_xlabel("k"); axes[1].set_ylabel("silhouette")

plt.tight_layout()
plt.savefig(f"clustering_predict_{name}_{mob3.shape[0]}.png", dpi=800)
plt.show()

best_k = list(k_range)[int(np.argmax(sils[1:])) + 1]
km_final = KMeans(n_clusters=best_k, random_state=42, n_init=10)
labels_final = km_final.fit_predict(Xt)
sil_final = silhouette_score(Xt, labels_final)

results = {name: {"X": Xt, "feature_names": feat_names, "k": best_k, "labels": labels_final, "silhouette": sil_final}}
print(f"{name}: best_k={best_k}, silhouette={sil_final:.3f}")

## PCA projection
Projects each version onto 2 principal components to visually inspect how well-separated the resulting clusters are.

In [ ]:
name, r = next(iter(results.items()))
X_df = pd.DataFrame(r["X"], columns=r["feature_names"])
pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(X_df)
k_r, sil_r = r["k"], r["silhouette"]

feat_names_in = pca.feature_names_in_
loadings = pca.components_
top_pc1 = feat_names_in[np.argmax(np.abs(loadings[0]))]
top_pc2 = feat_names_in[np.argmax(np.abs(loadings[1]))]
var_ratio = pca.explained_variance_ratio_

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(coords[:, 0], coords[:, 1], c=r["labels"], cmap="tab10", s=15, alpha=0.8)

centroids = np.array([coords[r["labels"] == i].mean(axis=0) for i in range(k_r)])
ax.scatter(centroids[:, 0], centroids[:, 1], c="black", marker="X", s=150,
           edgecolor="white", linewidth=1.5, label="Centroids")
ax.legend()

ax.set_xlabel(f"PC1 ({var_ratio[0]:.1%} var — top: {top_pc1})")
ax.set_ylabel(f"PC2 ({var_ratio[1]:.1%} var — top: {top_pc2})")
ax.set_title(f"{name} (k={k_r}, sil={sil_r:.2f})")
plt.tight_layout()
plt.savefig("clustering_comparison.png", dpi=800)
plt.show()

In [12]:
summary = pd.DataFrame({
    "version": list(results.keys()),
    "best_k": [r["k"] for r in results.values()],
    "silhouette": [round(r["silhouette"], 3) for r in results.values()],
}).sort_values("silhouette", ascending=False)
summary

,version,best_k,silhouette
1,geo_economic_focus,5,0.235
3,equity_focus,3,0.195
2,demo_field_focus,3,0.177
0,baseline,7,0.143


## Best version selection
Picks the version with the highest silhouette score and assigns the corresponding cluster labels to the main dataframe.

In [ ]:
best_version = summary.iloc[0]["version"]
mob3["cluster"] = results[best_version]["labels"]
best_version

## Per-sample silhouette plot
Shows the distribution of silhouette coefficients per individual sample, grouped by cluster.

In [ ]:
sil_vals = silhouette_samples(results[best_version]["X"], mob3["cluster"])
best_k_final = results[best_version]["k"]
fig, ax = plt.subplots(figsize=(7, 4))
y_lower = 0
for i in sorted(mob3["cluster"].unique()):
    vals = np.sort(sil_vals[mob3["cluster"] == i])
    y_upper = y_lower + len(vals)
    ax.fill_betweenx(np.arange(y_lower, y_upper), 0, vals, alpha=0.7, label=f"Cluster {i}")
    y_lower = y_upper
ax.axvline(sil_vals.mean(), color="red", linestyle="--", label="Mean silhouette")
ax.set_xlabel("Silhouette coefficient")
ax.set_ylabel("Samples (grouped by cluster)")
ax.set_title(f"Silhouette plot — {best_version}, k={best_k_final}")
ax.legend()
plt.tight_layout()
plt.savefig(f"silhouette_plot_{best_version}.png", dpi=800)
plt.show()

## Standardized profile heatmap
Z-score of each cluster's mean relative to the overall mean, for numeric features.

In [ ]:
z_profile = (mob3.groupby("cluster")[student_num_cols].mean() - mob3[student_num_cols].mean()) / mob3[student_num_cols].std()

fig, ax = plt.subplots(figsize=(8, 4))
sns.heatmap(z_profile, annot=True, fmt=".2f", cmap="RdBu_r", center=0, ax=ax)
ax.set_title("Standardized deviation from overall mean, by cluster")
plt.tight_layout()
plt.show()

## Cluster size
Count of observations per cluster, useful for correctly interpreting η² and Cramér’s V results 

In [ ]:
mob3["cluster"].value_counts().sort_index()

## Statistical tests across clusters
Eta squared method for numeric features.

In [ ]:
maroon_cmap = LinearSegmentedColormap.from_list('maroon_theme', ['white', '#591427'])

def eta_squared(*groups):
    all_data = np.concatenate(groups)
    grand_mean = all_data.mean()
    ss_between = sum(len(g) * (g.mean() - grand_mean)**2 for g in groups)
    ss_total = sum((all_data - grand_mean)**2)
    return ss_between / ss_total

eta_rows = []

for col in eta_cols:
    groups = [
        mob3.loc[mob3["cluster"] == c, col].dropna().values
        for c in sorted(mob3["cluster"].unique())
    ]

    eta2 = eta_squared(*groups)
    _, p_val = stats.f_oneway(*groups)

    eta_rows.append({
        "Feature": col,
        "η²": round(eta2, 4),
        "p-value": p_val
    })

df_eta = (
    pd.DataFrame(eta_rows)
    .sort_values("η²", ascending=False)
    .reset_index(drop=True)
)

df_eta.style.background_gradient(cmap=maroon_cmap, subset=["η²"]).format({"η²": "{:.4f}", "p-value": "{:.2e}"})

Cramer's V method for categorical features.

In [ ]:
def cramers_v(confusion_matrix):
    chi2, p, dof, expected = stats.chi2_contingency(confusion_matrix)
    n = confusion_matrix.sum().sum()
    r, k = confusion_matrix.shape
    v = np.sqrt(chi2 / (n * (min(r, k) - 1)))
    return v, p

cramers_rows = []

for col in student_cat_cols + ["Academic Year"]:
    ct = pd.crosstab(mob3["cluster"], mob3[col])

    if ct.shape[0] > 1 and ct.shape[1] > 1:
        v, p = cramers_v(ct)
    else:
        v, p = np.nan, np.nan

    cramers_rows.append({
        "Feature": col,
        "Cramér's V": round(v, 4) if not np.isnan(v) else np.nan,
        "p-value": p
    })

df_cramers = (
    pd.DataFrame(cramers_rows)
    .sort_values("Cramér's V", ascending=False)
    .reset_index(drop=True)
)

df_cramers.style.background_gradient(cmap=maroon_cmap, subset=["Cramér's V"]).format({"Cramér's V": "{:.4f}", "p-value": "{:.2e}"})

## Multiple testing correction
When testing many features simultaneously (via η² and Cramér’s V), the risk of false positives increases. We therefore apply FDR correction (Benjamini–Hochberg) to the p‑values in both tables.

In [ ]:
from statsmodels.stats.multitest import multipletests

df_eta["significant_FDR"] = multipletests(df_eta["p-value"], alpha=0.05, method="fdr_bh")[0]
df_cramers_valid = df_cramers.dropna(subset=["p-value"]).copy()
df_cramers_valid["significant_FDR"] = multipletests(df_cramers_valid["p-value"], alpha=0.05, method="fdr_bh")[0]
df_cramers = df_cramers.merge(df_cramers_valid[["Feature", "significant_FDR"]], on="Feature", how="left")

print("Significant η² after FDR:")
display(df_eta[["Feature", "η²", "p-value", "significant_FDR"]])

print("\nSignificant Cramér's V after FDR:")
display(df_cramers[["Feature", "Cramér's V", "p-value", "significant_FDR"]])

## Effect size interpretation
Adds qualitative labels to η² and Cramér’s V (per Cohen) to make effect sizes immediately interpretable, especially when statistical significance in mob3 may be misleading.

In [ ]:
def interpret_eta2(v):
    if pd.isna(v): return "n/a"
    if v < 0.01: return "negligible"
    if v < 0.06: return "small"
    if v < 0.14: return "medium"
    return "large"

def interpret_cramers_v(v):
    if pd.isna(v): return "n/a"
    if v < 0.10: return "negligible"
    if v < 0.30: return "small"
    if v < 0.50: return "medium"
    return "large"

df_eta["effect_size"] = df_eta["η²"].apply(interpret_eta2)
df_cramers["effect_size"] = df_cramers["Cramér's V"].apply(interpret_cramers_v)

display(df_eta[["Feature", "η²", "effect_size", "significant_FDR"]])
display(df_cramers[["Feature", "Cramér's V", "effect_size", "significant_FDR"]])

## Combined feature ranking
“Combines η² and Cramér’s V into a unified ranking (both normalized to [0, 1]) to identify which variables most strongly define the clusters, regardless of whether they are numerical or categorical.

In [ ]:
combined = pd.concat([
    df_eta.rename(columns={"η²": "effect_value"})[["Feature", "effect_value", "effect_size", "significant_FDR"]].assign(type="numeric (η²)"),
    df_cramers.rename(columns={"Cramér's V": "effect_value"})[["Feature", "effect_value", "effect_size", "significant_FDR"]].assign(type="categorical (Cramér's V)"),
], ignore_index=True).sort_values("effect_value", ascending=False).reset_index(drop=True)

combined.style.background_gradient(cmap=maroon_cmap, subset=["effect_value"]).format({"effect_value": "{:.4f}"})

## Post-hoc (numerical)
η² indicates how strongly a feature separates the clusters overall, but not which specific cluster pairs differ. Tukey’s HSD on the numerical feature with the highest η² provides the required pairwise comparisons.

In [ ]:
from statsmodels.stats.multicomp import pairwise_tukeyhsd

top_numeric_feature = df_eta.iloc[0]["Feature"]
print(f"Numeric feature with the strongest effect: {top_numeric_feature}")

tukey = pairwise_tukeyhsd(
    endog=mob3[top_numeric_feature].dropna(),
    groups=mob3.loc[mob3[top_numeric_feature].notna(), "cluster"],
    alpha=0.05
)
print(tukey)

## Post-hoc (categorical)
“Cramér’s V shows overall association, while standardized residuals reveal which cluster–category cells drive it (|residual| > 2 ≈ significant).

In [ ]:
top_cat_feature = df_cramers.dropna(subset=["Cramér's V"]).iloc[0]["Feature"]
print(f"Categorical feature with the strongest effect: {top_cat_feature}")

ct = pd.crosstab(mob3["cluster"], mob3[top_cat_feature])
chi2, p, dof, expected = stats.chi2_contingency(ct)
std_resid = (ct - expected) / np.sqrt(expected)

fig, ax = plt.subplots(figsize=(max(6, ct.shape[1]*0.8), 4))
sns.heatmap(std_resid, annot=True, fmt=".1f", cmap="RdBu_r", center=0, ax=ax)
ax.set_title(f"Standardized residuals — cluster vs {top_cat_feature}")
plt.tight_layout()
plt.show()

## Cluster stability check
Evaluates cluster‑assignment stability by refitting k‑means (same k) on a bootstrap resample and comparing labels via the Adjusted Rand Index (ARI). Values near 1 indicate a robust partition not driven by sampling noise.

In [ ]:
from sklearn.metrics import adjusted_rand_score

n_boot = 20
ari_scores = []
X_best = results[best_version]["X"]
k_best = results[best_version]["k"]
labels_orig = mob3["cluster"].values

rng = np.random.default_rng(42)
for i in range(n_boot):
    idx = rng.choice(len(X_best), size=len(X_best), replace=True)
    km_boot = KMeans(n_clusters=k_best, random_state=i, n_init=10)
    labels_boot = km_boot.fit_predict(X_best[idx])
    ari_scores.append(adjusted_rand_score(labels_orig[idx], labels_boot))

print(f"Mean ARI over {n_boot} bootstrap resamples: {np.mean(ari_scores):.3f} (± {np.std(ari_scores):.3f})")

plt.figure(figsize=(6, 4))
plt.hist(ari_scores, bins=10, color="#591427", edgecolor="white")
plt.xlabel("Adjusted Rand Index")
plt.ylabel("Frequency")
plt.title("Clustering stability (bootstrap)")
plt.tight_layout()
plt.show()

## Cross-validating version selection: Calinski-Harabasz & Davies-Bouldin

In [ ]:
from sklearn.metrics import calinski_harabasz_score, davies_bouldin_score

validation_rows = []
for name, r in results.items():
    ch = calinski_harabasz_score(r["X"], r["labels"])
    db = davies_bouldin_score(r["X"], r["labels"])
    validation_rows.append({
        "version": name,
        "k": r["k"],
        "silhouette": round(r["silhouette"], 3),
        "calinski_harabasz": round(ch, 1),
        "davies_bouldin": round(db, 3),
    })

df_validation = pd.DataFrame(validation_rows)
df_validation["rank_silhouette"] = df_validation["silhouette"].rank(ascending=False)
df_validation["rank_calinski_harabasz"] = df_validation["calinski_harabasz"].rank(ascending=False)
df_validation["rank_davies_bouldin"] = df_validation["davies_bouldin"].rank(ascending=True)
df_validation.sort_values("rank_silhouette")